# Jacobians & Vector Calculus

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/04-jacobians

A from-scratch, runnable derivation of Jacobians for the layers you use every day.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — the gradient, generalized

A gradient describes a function with **one** output. A **Jacobian** describes a function
with *many* outputs: for `f: ℝⁿ → ℝᵐ` it's the `m×n` matrix stacking one gradient per
output, `J_ij = ∂f_i/∂x_j`. It's the **best linear approximation** of the map near a
point (`f(x+δ) ≈ f(x) + Jδ`), and it's the object backpropagation multiplies through: a
network is a chain of layers, and the chain rule says its Jacobian is the *product* of
the per-layer Jacobians. We'll derive the Jacobians of the layers you use daily — affine,
element-wise, softmax — from scratch, then confirm them with autodiff.

## The Jacobian: definition and shape

The Jacobian of $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ is the $m \times n$ matrix $J_{ij} = \partial f_i / \partial x_j$.
It linearly approximates $\mathbf{f}$ near any point: $\mathbf{f}(\mathbf{x}+\boldsymbol{\delta}) \approx \mathbf{f}(\mathbf{x}) + \mathbf{J}\boldsymbol{\delta}$.

We derive each Jacobian **by hand** (pure stdlib) and confirm it with a **finite-difference** estimate; §2 then cross-checks everything against **autodiff** (`jax.jacobian`).

In [ ]:
import math

# --- Affine layer: f(x) = Wx + b ---
# Jacobian should equal W exactly
W = [[2.0, -1.0, 0.5],
     [0.0,  1.0, 3.0]]   # 2 outputs, 3 inputs
b = [1.0, -0.5]

def affine(x):
    return [sum(W[i][j] * x[j] for j in range(3)) + b[i] for i in range(2)]

# Finite-difference Jacobian (central differences, h=1e-5)
def fd_jacobian(f, x, h=1e-5):
    n = len(x)
    y0 = f(x)
    m = len(y0)
    J = [[0.0]*n for _ in range(m)]
    for j in range(n):
        xp = x[:]; xp[j] += h
        xm = x[:]; xm[j] -= h
        fp = f(xp); fm = f(xm)
        for i in range(m):
            J[i][j] = (fp[i] - fm[i]) / (2*h)
    return J

x0 = [1.0, 2.0, 3.0]
J_fd = fd_jacobian(affine, x0)

print('Finite-difference Jacobian of affine layer:')
for row in J_fd:
    print(' ', [round(v, 6) for v in row])
print('Weight matrix W:')
for row in W:
    print(' ', row)

# The Jacobian of an affine map is exactly its weight matrix W
assert all(abs(J_fd[i][j] - W[i][j]) < 1e-8 for i in range(2) for j in range(3)), \
    'Jacobian of affine layer must equal W'
print('VERIFY: J_affine == W:', True)

**What to notice:** the finite-difference Jacobian of the affine layer equals `W`
exactly. That's the simplest and most important case: **a linear layer's Jacobian is just
its weight matrix** — no matter the input. Everything else builds on this.

## Element-wise activations: diagonal Jacobian

For $\mathbf{f}(\mathbf{x}) = (\phi(x_1), \ldots, \phi(x_n))$ (same scalar $\phi$ applied independently), output $i$ only depends on input $i$, so $J_{ij} = \phi'(x_i) \cdot \mathbb{1}[i=j]$. The Jacobian is diagonal. Verify for ReLU and tanh.

In [ ]:
def relu(x): return [max(0.0, xi) for xi in x]
def tanh_vec(x): return [math.tanh(xi) for xi in x]

x0 = [-1.5, 0.5, 2.0, -0.3]
J_relu = fd_jacobian(relu, x0)
J_tanh = fd_jacobian(tanh_vec, x0)

# Off-diagonal entries must be ~0
for name, J in [('ReLU', J_relu), ('tanh', J_tanh)]:
    n = len(x0)
    off = sum(abs(J[i][j]) for i in range(n) for j in range(n) if i != j)
    diag = [J[i][i] for i in range(n)]
    print(f'{name}: diagonal = {[round(d,4) for d in diag]}, off-diagonal sum = {off:.2e}')

# ReLU diagonal: 1 where x>0, 0 elsewhere
relu_diag = [J_relu[i][i] for i in range(4)]
assert [round(d) for d in relu_diag] == [0, 1, 1, 0], \
    'ReLU Jacobian diagonal should be indicator of x>0'

# tanh diagonal: 1-tanh(xi)^2 (the sech^2 derivative)
tanh_diag = [J_tanh[i][i] for i in range(4)]
expected_tanh_diag = [1 - math.tanh(xi)**2 for xi in x0]
assert all(abs(tanh_diag[i] - expected_tanh_diag[i]) < 1e-6 for i in range(4)), \
    'tanh Jacobian diagonal should be 1 - tanh(x)^2'

print('VERIFY: element-wise activations have diagonal Jacobians.')

**What to notice:** for a per-element activation the off-diagonal sum is ~`1e-16` — the
Jacobian is **diagonal**, because output `i` depends only on input `i`. ReLU's diagonal is
the 0/1 gate `𝟙[x>0]`; tanh's is `1 − tanh²`. Backprop through an activation is therefore
just an element-wise multiply, not a full matrix product.

## Softmax Jacobian: the coupled case

Softmax is not independent per output — all outputs share the same normaliser. The Jacobian is dense:
$\mathbf{J} = \operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.

Verify numerically for the two-class case $\mathbf{x} = (1, 0)$ from the lesson.

In [ ]:
def softmax(x):
    e = [math.exp(xi - max(x)) for xi in x]   # numerically stable
    z = sum(e)
    return [ei / z for ei in e]

x0 = [1.0, 0.0]
s = softmax(x0)
print('softmax([1, 0]) =', [round(si, 4) for si in s])

J_sm_fd   = fd_jacobian(softmax, x0)
J_sm_anal = [[s[i]*(1 - s[j]) if i == j else -s[i]*s[j]
              for j in range(2)] for i in range(2)]

print('FD Jacobian:       ', [[round(v, 4) for v in row] for row in J_sm_fd])
print('Analytic Jacobian: ', [[round(v, 4) for v in row] for row in J_sm_anal])

assert all(abs(J_sm_fd[i][j] - J_sm_anal[i][j]) < 1e-7 for i in range(2) for j in range(2)), \
    'FD and analytic softmax Jacobians should agree'

# Rows must sum to 0 (probabilities sum to 1)
row_sums = [sum(J_sm_anal[i]) for i in range(2)]
assert all(abs(s) < 1e-12 for s in row_sums), 'Rows of softmax Jacobian must sum to 0'
print('VERIFY: rows sum to zero (probabilities are constrained to sum to 1).')

**What to notice:** softmax is the exception — its Jacobian `diag(σ) − σσᵀ` is **dense**,
because every output shares the normalizer, and its **rows sum to 0** (nudging a logit
can't change the total probability, which is pinned at 1). That zero-row-sum makes the
Jacobian singular — a point we return to in the gotchas.

## Chain rule: Jacobian of a composed network

For $\mathbf{h} = \mathbf{g} \circ \mathbf{f}$, the Jacobian is the **product** $\mathbf{J}_{\mathbf{g}} \cdot \mathbf{J}_{\mathbf{f}}$ (computed at the appropriate points). Verify for a two-layer network: linear → ReLU → linear.

In [ ]:
# Two-layer network: R^3 -> R^2 -> R^2 -> R^1
# Layer 1: R^3 -> R^2 (affine + ReLU)
# Layer 2: R^2 -> R^1 (affine, no activation)

W1 = [[1.0, -0.5, 0.3], [0.2, 1.0, -1.0]]; b1 = [0.1, -0.2]
W2 = [[0.8, -0.6]]; b2 = [0.0]

def layer1(x): return [max(0.0, sum(W1[i][j]*x[j] for j in range(3)) + b1[i]) for i in range(2)]
def layer2(y): return [sum(W2[i][j]*y[j] for j in range(2)) + b2[i] for i in range(1)]
def network(x): return layer2(layer1(x))

x0 = [0.5, -1.0, 2.0]

# Full Jacobian by FD
J_full = fd_jacobian(network, x0)

# Via chain rule: J_L2 @ J_L1 (each computed at the appropriate intermediate point)
z1 = layer1(x0)
J1 = fd_jacobian(layer1, x0)     # 2x3
J2 = fd_jacobian(layer2, z1)     # 1x2

# matrix multiply J2 (1x2) @ J1 (2x3) -> (1x3)
J_chain = [[sum(J2[i][k]*J1[k][j] for k in range(2)) for j in range(3)] for i in range(1)]

print('J via FD on full network:', [[round(v, 6) for v in row] for row in J_full])
print('J via chain rule:        ', [[round(v, 6) for v in row] for row in J_chain])

assert all(abs(J_full[i][j] - J_chain[i][j]) < 1e-7
           for i in range(1) for j in range(3)), 'Chain rule must equal direct FD'
print('VERIFY: J_network = J_layer2 @ J_layer1 (vector chain rule).')

**What to notice:** the Jacobian of the whole network equals `J_layer2 @ J_layer1` — the
vector chain rule is a **matrix product** of per-layer Jacobians. This is exactly what
backprop evaluates; the only trick (next) is doing it without ever forming the full
matrices.

## 2. The library way — `jax.jacobian`

Autodiff computes Jacobians directly. `jax.jacobian(f)(x)` returns the full `m×n` matrix
by differentiating `f`'s operations — no hand calculus, no finite-difference noise. The
cell confirms it reproduces the affine and softmax Jacobians from §1. (Under the hood
`jax.jacrev` is *reverse mode* — the same direction as backprop — while `jax.jacfwd` is
forward mode; more on that in the gotchas.)

In [ ]:
import jax, jax.numpy as jnp

# affine: J should equal W exactly
Wj, bj = jnp.array(W), jnp.array(b)
J_affine_ad = np.array(jax.jacobian(lambda x: Wj @ x + bj)(jnp.array([1.0, 2.0, 3.0])))
assert np.allclose(J_affine_ad, np.array(W)), "autodiff affine Jacobian must equal W"

# softmax: J should equal diag(sigma) - sigma sigma^T
def softmax_j(x):
    e = jnp.exp(x - x.max()); return e / e.sum()
xs = jnp.array([1.0, 0.0])
J_softmax_ad = np.array(jax.jacobian(softmax_j)(xs))
sig = np.array(softmax_j(xs))
J_softmax_anal = np.diag(sig) - np.outer(sig, sig)

print('autodiff softmax Jacobian:\n', J_softmax_ad.round(4))
print('analytic softmax Jacobian:\n', J_softmax_anal.round(4))
assert np.allclose(J_softmax_ad, J_softmax_anal, atol=1e-6), "autodiff must match analytic softmax J"
print('\nautodiff Jacobians match the by-hand derivations (affine + softmax) ✓')

**What to notice:** autodiff reproduces both Jacobians to machine precision. In real code
you'd never derive these by hand — but knowing the shapes (`J = W`, diagonal, `diag(σ)−σσᵀ`)
lets you *sanity-check* autodiff and understand its memory/cost behavior.

## VJP vs full Jacobian — cost comparison

Backpropagation computes $\mathbf{v}^\top \mathbf{J}$ (vector-Jacobian product) in a single backward pass, rather than computing all $mn$ entries of $\mathbf{J}$. For large $n, m$, this is the difference between feasibility and impossibility.

In [ ]:
import time

for n, m in [(100, 100), (500, 500), (1000, 1000)]:
    W_large = np.random.randn(m, n) * 0.1
    x_large = np.random.randn(n)
    v_large = np.random.randn(m)

    # VJP: v^T J = v^T W = W^T v  (one matrix-vector multiply)
    t0 = time.perf_counter()
    for _ in range(200):
        vjp = W_large.T @ v_large
    t_vjp = (time.perf_counter() - t0) / 200 * 1e6

    # Full Jacobian is just W (exact), but for a non-linear layer we'd need m forward passes
    # Simulate the cost: m forward passes through a vector of length n
    t0 = time.perf_counter()
    for _ in range(50):
        J_full_sim = np.zeros((m, n))
        for i in range(min(m, 20)):   # only 20 rows to keep this fast
            e_i = np.zeros(m); e_i[i] = 1.0
            J_full_sim[i] = W_large.T @ e_i  # proxy for one JVP
    t_full = (time.perf_counter() - t0) / 50 * 1e6 * (m / 20)

    print(f'n={n}, m={m}: VJP ≈ {t_vjp:.1f} µs | full Jacobian ≈ {t_full:.0f} µs (est.)')

print()
print('VJP cost is O(n), full Jacobian is O(mn) — the key reason backprop is efficient.')

**What to notice:** the vector-Jacobian product `vᵀJ` stays cheap (`O(n)`, one
matrix-vector multiply) while materializing the full Jacobian grows as `O(mn)` and pulls
away fast. Backprop *never* forms `J` — it propagates `vᵀJ` layer by layer, which is the
whole reason training billion-parameter models is even possible.

## Gain vs conditioning — what initialization actually controls

Two different numbers describe a layer's Jacobian `W`. Its **gain** — the largest
singular value `σ_max = ‖W‖`, how much it can stretch a vector — decides whether signals
**grow or shrink** as they pass through many layers. Its **condition number**
`κ = σ_max/σ_min` — the ratio of most to least stretch — measures *direction-dependent*
distortion. Here's the subtlety usually gotten wrong: **scaling every weight by a constant
changes the gain but leaves `κ` unchanged** (all singular values scale together). So
initialization (Xavier/He) tunes the **gain** toward 1 to keep activations and gradients
stable across depth — it does **not** fix the condition number, which depends on the
matrix's *shape*, not its scale. The next cell demonstrates exactly that with one matrix
at two scales.

In [ ]:
def sigma_max(W):                 # gain = operator norm = largest singular value
    return np.linalg.svd(W, compute_uv=False)[0]

def cond(W):
    sv = np.linalg.svd(W, compute_uv=False)
    return sv[0] / sv[-1]

np.random.seed(1)
d = 64
gains_xavier, gains_large, kappa_equal = [], [], []
for _ in range(6):
    Z  = np.random.randn(d, d)                 # ONE matrix, two scalings
    Wx = Z * np.sqrt(2.0 / (d + d))            # Xavier scale
    Wl = Z * 2.0                               # large scale
    gains_xavier.append(sigma_max(Wx))
    gains_large.append(sigma_max(Wl))
    kappa_equal.append(abs(cond(Wx) - cond(Wl)) < 1e-6)   # same κ?

fig, ax = plt.subplots()
ax.plot(gains_xavier, 'o-', color='#6366f1', label='Xavier gain σ_max')
ax.plot(gains_large,  's--', color='#f59e0b', label='Large-init gain σ_max')
ax.axhline(1.0, color='#2dd4bf', lw=1, ls=':', label='gain = 1 (ideal)')
ax.set_xlabel('layer index'); ax.set_ylabel('gain  σ_max(W)')
ax.set_title('Layer gain: Xavier stays near 1, large init explodes')
ax.legend(); plt.show()

print('mean gain: Xavier {:.2f}  vs  large {:.2f}'.format(
    np.mean(gains_xavier), np.mean(gains_large)))
print('condition number identical across the two scalings?', all(kappa_equal))

**What to notice:** same matrix, two scales — the Xavier gain hugs 1 while the large-init
gain sits ~16× higher (signals would blow up across depth). Yet the **condition number is
identical** for both scalings (`True`), proving `κ` is scale-invariant. That's the whole
point: initialization sets the *gain*, not the conditioning. He/Xavier keep per-layer gain
≈ 1 so the product across depth neither vanishes nor explodes.

## 4. Gotchas & mode choice

- **The softmax Jacobian is singular.** Its rows sum to 0, so it has rank `n−1` and no
  inverse — you can't "divide out" a softmax. (Cross-entropy is designed so the combined
  gradient stays clean anyway: `σ − y`.)
- **Forward vs reverse mode.** `jacrev` (reverse mode = backprop) costs one pass per
  *output* — cheap when `n ≫ m` (many inputs, few outputs, e.g. a scalar loss). `jacfwd`
  (forward mode) costs one pass per *input* — cheaper when `m ≫ n`. ML losses are scalar,
  so reverse mode wins, which is why backprop is reverse-mode autodiff.
- **Conditioning governs stability.** `κ(J) = σ_max/σ_min`; as it blows up, gradient flow
  splits into exploding and vanishing directions.

In [ ]:
# 1) softmax Jacobian is singular (rows sum to zero -> rank n-1)
sig = np.array([0.5, 0.3, 0.2])
J = np.diag(sig) - np.outer(sig, sig)
print('row sums   :', J.sum(axis=1).round(12), '-> all zero')
print('rank / size:', np.linalg.matrix_rank(J), '/', len(sig), '-> singular, no inverse')

# 2) reverse vs forward mode: a scalar-loss gradient is one reverse pass, n forward passes
import jax, jax.numpy as jnp
loss = lambda x: jnp.sum(x**2)                      # R^n -> R (m=1)
x = jnp.arange(1.0, 5.0)
print('\njacrev (reverse, backprop):', np.array(jax.jacrev(loss)(x)))   # 1 pass
print('jacfwd (forward)          :', np.array(jax.jacfwd(loss)(x)),
      ' <- would need n passes for a wide Jacobian')

**What to notice:** the softmax Jacobian's rows sum to exactly 0, giving rank 2 out of 3 —
provably singular. And both autodiff modes return the same gradient `[2,4,6,8]`, but for a
scalar loss `jacrev` gets it in **one** backward pass while `jacfwd` would need one pass
per input — the concrete reason ML uses reverse-mode autodiff.

## Key takeaways

- **Jacobian shape**: $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ → $\mathbf{J} \in \mathbb{R}^{m \times n}$ (outputs × inputs).
- **Affine layer**: $\mathbf{J} = \mathbf{W}$; **element-wise**: diagonal; **softmax**: $\operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.
- **Vector chain rule**: $\mathbf{J}_{\mathbf{g}\circ\mathbf{f}} = \mathbf{J}_{\mathbf{g}} \cdot \mathbf{J}_{\mathbf{f}}$ — backprop is this product computed efficiently.
- **VJP** $\mathbf{v}^\top \mathbf{J}$ costs $O(n)$; full Jacobian costs $O(mn)$ — the key to scalable backprop.
- **Gain** $\sigma_{\max}(\mathbf{J})$ governs whether signals grow or shrink across depth; Xavier/He initialization targets gain $\approx 1$. The **condition number** $\kappa$ is scale-invariant — it measures direction-dependent distortion, not something init controls.

## ✏️ Your turn

### Exercise 1 — Jacobian of the softmax (from scratch)

The softmax Jacobian is $\mathbf{J} = \operatorname{diag}(\boldsymbol{\sigma}) - \boldsymbol{\sigma}\boldsymbol{\sigma}^\top$.

Implement it and verify the key structural properties: rows sum to 0, symmetry, and exact match with a finite-difference estimate.

In [ ]:
import numpy as np

def softmax_jacobian(x):
    """Analytic Jacobian of softmax at x (1-D array).
    Returns J of shape (n, n) where J = diag(sigma) - sigma @ sigma.T."""
    # TODO(you): compute softmax probabilities sigma, then build the Jacobian
    ...

In [ ]:
x = np.array([1.0, 2.0, 0.5])
J = softmax_jacobian(x)

assert J.shape == (3, 3), "Jacobian must be (n, n)"
assert np.allclose(J.sum(axis=1), 0.0, atol=1e-9), \
    "rows of the softmax Jacobian must sum to 0 (probabilities sum to 1)"
assert np.allclose(J, J.T, atol=1e-12), \
    "softmax Jacobian is symmetric"

# Finite-difference check
def sm(x): e = np.exp(x - x.max()); return e / e.sum()
h = 1e-5
J_fd = np.zeros((3, 3))
for j in range(3):
    xp, xm = x.copy(), x.copy()
    xp[j] += h; xm[j] -= h
    J_fd[:, j] = (sm(xp) - sm(xm)) / (2*h)
assert np.allclose(J, J_fd, atol=1e-6), \
    "analytic Jacobian must match finite-difference estimate"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def softmax_jacobian(x):
    e = np.exp(x - x.max())
    sigma = e / e.sum()
    return np.diag(sigma) - np.outer(sigma, sigma)
```

</details>

### Exercise 2 — VJP of the affine layer

For $\mathbf{y} = \mathbf{W}\mathbf{x} + \mathbf{b}$ with $\mathbf{W} \in \mathbb{R}^{m \times n}$, the Jacobian is $\mathbf{W}$.
The VJP with upstream gradient $\mathbf{v} \in \mathbb{R}^m$ is $\mathbf{v}^\top \mathbf{J} = \mathbf{v}^\top \mathbf{W}$, which equals $\mathbf{W}^\top \mathbf{v}$ as a column vector.

Implement it and verify it matches the finite-difference gradient.

In [ ]:
def affine_vjp(W, x, b, v):
    """VJP of y = Wx + b with upstream gradient v.
    Returns grad_x = W^T v (same shape as x)."""
    # TODO(you): one line
    ...

In [ ]:
import numpy as np

np.random.seed(7)
m_dim, n_dim = 4, 6
W2 = np.random.randn(m_dim, n_dim)
b2 = np.random.randn(m_dim)
x2 = np.random.randn(n_dim)
v2 = np.random.randn(m_dim)

grad_x = affine_vjp(W2, x2, b2, v2)

assert grad_x.shape == x2.shape, \
    "VJP output must have same shape as x"
assert np.allclose(grad_x, W2.T @ v2, atol=1e-12), \
    "VJP must equal W^T @ v"

# Finite-difference check: grad_x[j] = d/dx_j (v^T (Wx+b))
loss = lambda x: float(v2 @ (W2 @ x + b2))
h = 1e-5
grad_fd = np.array([(loss(x2 + h*np.eye(n_dim)[j]) - loss(x2 - h*np.eye(n_dim)[j])) / (2*h)
                    for j in range(n_dim)])
assert np.allclose(grad_x, grad_fd, atol=1e-7), \
    "VJP must match finite-difference gradient of v^T(Wx+b) w.r.t. x"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def affine_vjp(W, x, b, v):
    return W.T @ v
```

</details>

### Exercise 3 — Layer gain and the scale-invariance of κ

A layer's **gain** is the largest singular value of its weight matrix,
$\sigma_{\max}(W) = \lVert W \rVert$ — the most it can stretch any input. Initialization
tunes this toward 1. Implement it, and the checks confirm the key fact: scaling a matrix
scales its **gain** but leaves its **condition number** unchanged.

In [ ]:
import numpy as np

def spectral_gain(W):
    """Largest singular value σ_max(W) = ||W|| (the operator norm / gain)."""
    # TODO(you): singular values via np.linalg.svd(W, compute_uv=False), return the max
    ...

In [ ]:
np.random.seed(42)
d = 64
Z = np.random.randn(d, d)

assert abs(spectral_gain(np.eye(d)) - 1.0) < 1e-9, \
    "the identity has unit gain (all singular values equal 1)"

gain_small = spectral_gain(Z * np.sqrt(2.0 / (d + d)))   # Xavier scale
gain_big   = spectral_gain(Z * 2.0)                       # large scale
assert gain_big > gain_small, "a larger weight scale gives a larger gain"
assert abs(gain_big / gain_small - 2.0 / np.sqrt(2.0 / (d + d))) < 1e-6, \
    "gain scales exactly with the weight scale"

cond = lambda W: (lambda sv: sv[0] / sv[-1])(np.linalg.svd(W, compute_uv=False))
assert abs(cond(Z * 0.1) - cond(Z * 10.0)) < 1e-6, \
    "condition number is scale-invariant: scaling W leaves κ unchanged"
print(f"gain(Xavier) = {gain_small:.2f} | gain(large) = {gain_big:.2f} | "
      f"κ is identical across scales")
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def spectral_gain(W):
    return np.linalg.svd(W, compute_uv=False)[0]
```

</details>